# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  
**Ciara Graves**

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    for attempt in range(tries):
        # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
        r = requests.get(BASE, params=params, timeout=10)
        if r.status_code == 200:
            return pd.DataFrame(r.json()['daily'])
        # TODO:   raise on a status that is not in RETRY_STATUSES
        if r.status_code not in RETRY_STATUSES:
            r.raise_for_status()
        # TODO:   otherwise sleep (increasing each time) and try again
        if attempt < tries - 1:
            time.sleep(2 ** attempt)
    # TODO: raise if every attempt failed
    raise RuntimeError("Weather request failed after all attempts")


**Q1 Explanation:** Here, I reused the code block created in this week's studio to pull the weather data from open-meteo.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
#       then keep just the two dates you care about
start_date = '2026-09-05'
end_date = '2026-10-03'
# Call API once
weather = get_weather(start_date, end_date)
# Keep only the two dates we care about
weather = weather[weather['time'].isin([start_date, end_date])]

used_fallback = False
print(weather)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


**Q2 Explanation:** By specifying the days or interest, we are able to pull all weather data within the interval in one request. This is helpful for both ease in running the code once, but also to limit pull requests.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
# If fallback was used
if used_fallback:
    wx = FALLBACK
    wx['source'] = 'fallback sample'
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'
print(wx)

          time  temperature_2m_max  precipitation_sum      source
0   2026-09-05                29.4                2.5  open-meteo
28  2026-10-03                22.0               14.7  open-meteo


**Q2 Explanation:** In this block, we built the fallback and labeled it as such. Here on out, if the fallback is used, it will output as such. This is helpful so the notebook is able to run locally/disconnected from any network; while also logging that the values are not necessarily real.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
# Rename cols
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
# Convert date to datetime
wx['date'] = pd.to_datetime(wx['date'])
# keep only specified cols
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]
print(wx)
#print(wx.dtypes) # to verify

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


**Q5 Explanation:** Here we created our own frame with specific column names, not what is taken directly from the API. This will allow for use to perform data manipulation later on.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


**Q5 Explanation:** All asserts pass and the weather table is deemed safe to join.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
# find max and min for wet and dry
wetter = wx.loc[wx['precip_mm'].idxmax()]
drier = wx.loc[wx['precip_mm'].idxmin()]
# Calc diff
diff = wetter['precip_mm'] - drier['precip_mm']
# output
print(f"{wetter['date']:%B %d, %Y} was the wetter date with {wetter['precip_mm']} mm of rain.")
print(f"{drier['date']:%B %d, %Y} was the drier date with {drier['precip_mm']} mm of rain.")

October 03, 2026 was the wetter date with 14.7 mm of rain.
September 05, 2026 was the drier date with 2.5 mm of rain.


**Q6 Explanation:** Of our two dates of interes, we have that October 03, 2026 was the wetter day with 14.7 mm of precipitation, and September 05, 2026 was the drier day with 2.5mm of precipitation. The code for the printed output can be generalized to any range of dates as well.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
# load-in dt
from datetime import datetime
lat, lon = 38.03, -78.51
print(f"Endpoint: {BASE}")
print(f"Coordinates: {lat}, {lon}")
print(f"Date range: {start_date} to {end_date}")
print(f"Pulled on: {datetime.now()}")
print(f"Data source: {wx['source'].iloc[0]}")

Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled on: 2026-10-12 02:07:24.345336
Data source: open-meteo


**Q7 Explanation:** Here we are able to output the endpoint API, coordinates, date range, pulled time, and data source. This is good for anyone reading the notebook to see when, where, and how the data was collected.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If `get_weather` returned an empty DataFrame on failure, rather than raising an error, then we would still be able to merge 200 days of sales and chart rain against revenue. However, the columns for weather data would be missing data and would be filled with NaNs, so this failure would become obvious when the data are plotted or statistics on weather-specific variables are calculated. However, no errors would be raised throughout this proccess. This emphasizes the need for EDA and code checks.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
# Specify cites in new col
charlottesville = wx.copy()
charlottesville['city'] = 'Charlottesville'
# Pull Orlando weather
orlando = get_weather('2026-09-05', '2026-10-03',
                      lat=28.54, lon=81.38)
orlando = orlando.rename(columns={
    'time':'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
# add source and city info
orlando['source']='open-meteo'
orlando['city']='Orlando'

# combine
combined_df = pd.concat(
    [charlottesville, orlando], ignore_index=True)
# maintain order
combined_df = combined_df[['city', 'date', 'temp_max', 'source']]
print(combined_df)

               city                 date  temp_max      source
0   Charlottesville  2026-09-05 00:00:00      29.4  open-meteo
1   Charlottesville  2026-10-03 00:00:00      22.0  open-meteo
2           Orlando           2026-09-05      29.8  open-meteo
3           Orlando           2026-09-06      32.4  open-meteo
4           Orlando           2026-09-07      30.4  open-meteo
5           Orlando           2026-09-08      31.3  open-meteo
6           Orlando           2026-09-09      31.6  open-meteo
7           Orlando           2026-09-10      31.7  open-meteo
8           Orlando           2026-09-11      30.4  open-meteo
9           Orlando           2026-09-12      32.2  open-meteo
10          Orlando           2026-09-13      31.3  open-meteo
11          Orlando           2026-09-14      30.9  open-meteo
12          Orlando           2026-09-15      30.6  open-meteo
13          Orlando           2026-09-16      30.3  open-meteo
14          Orlando           2026-09-17      30.3  ope

**Stretch Explanation:** Here we created the long format data, which is the same shape that we will use later on for our midterm project. It was interesting to join the two cities data together.